# RAG

Notebook initialization.

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

print("✅ Imports OK")

In [ ]:
# Base RAG = tickets résolus sauvegardés en Phase 3
rag_df = pd.read_csv(f"../{params['data']['rag_path']}")

print(f"Base RAG totale : {len(rag_df)} tickets résolus")
print(f"\nDistribution par catégorie :")
print(rag_df['Catégorie'].value_counts().to_string())

# Vérification que text_rag est présent et non-vide
vides = (rag_df['text_rag'].isna() | (rag_df['text_rag'] == '')).sum()
print(f"\nTickets avec text_rag vide : {vides}")

print(f"\nAperçu text_rag :")
print(rag_df[['Catégorie', 'text_rag', 'Résolution/Motif']]
      .head(3).to_string())

In [ ]:
class RAGEngine:
    """
    Moteur de recommandation par similarité cosinus.
    Suggère les résolutions historiques les plus proches
    d'un nouveau ticket entrant.
    """

    def __init__(self, top_k=3, similarity_threshold=0.30):
        self.top_k               = top_k
        self.similarity_threshold = similarity_threshold
        self.vectorizer          = None
        self.tfidf_matrix        = None
        self.knowledge_base      = None

    def fit(self, df: pd.DataFrame):
        """
        Construit la base de connaissance à partir des tickets résolus.
        df doit contenir : text_rag, Catégorie, Résolution/Motif, knowledge Base
        """
        # Filtrer les lignes avec text_rag non-vide
        self.knowledge_base = df[
            df['text_rag'].notna() & (df['text_rag'] != '')
        ].reset_index(drop=True)

        # Vectoriser la base de connaissance
        self.vectorizer = TfidfVectorizer(
            max_features = 5000,
            ngram_range  = (1, 2),
            sublinear_tf = True
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(
            self.knowledge_base['text_rag']
        )

        print(f"✅ RAG Engine fitted")
        print(f"   Base de connaissance : {len(self.knowledge_base)} tickets")
        print(f"   Matrice TF-IDF       : {self.tfidf_matrix.shape}")
        return self

    def recommend(self, query_text: str,
                  category_filter: str = None) -> list:
        """
        Retourne les top_k tickets les plus similaires.

        Args:
            query_text      : texte nettoyé du nouveau ticket
            category_filter : si fourni, filtre par catégorie d'abord

        Returns:
            liste de dicts avec score, résolution, catégorie
        """
        if not query_text or query_text.strip() == '':
            return []

        # Vectoriser la requête
        query_vec = self.vectorizer.transform([query_text])

        # Filtrage par catégorie (optionnel mais recommandé)
        if category_filter:
            mask = self.knowledge_base['Catégorie'] == category_filter
            if mask.sum() == 0:
                # Si pas de tickets dans cette catégorie → cherche partout
                mask = pd.Series([True] * len(self.knowledge_base))
        else:
            mask = pd.Series([True] * len(self.knowledge_base))

        kb_filtered   = self.knowledge_base[mask].reset_index(drop=True)
        matrix_filtered = self.tfidf_matrix[mask.values]

        # Similarité cosinus
        scores = cosine_similarity(query_vec, matrix_filtered).flatten()

        # Top K au-dessus du seuil
        top_indices = scores.argsort()[::-1][:self.top_k]

        results = []
        for idx in top_indices:
            score = float(scores[idx])
            if score < self.similarity_threshold:
                continue

            row = kb_filtered.iloc[idx]
            results.append({
                'score'      : round(score, 4),
                'categorie'  : row['Catégorie'],
                'resolution' : str(row['Résolution/Motif']),
                'knowledge'  : str(row['knowledge Base']),
                'titre'      : str(row['Titre'])
            })

        return results

    def save(self, path: str):
        joblib.dump(self, path)
        print(f"✅ RAG Engine sauvegardé : {path}")

    @staticmethod
    def load(path: str):
        engine = joblib.load(path)
        print(f"✅ RAG Engine chargé : {path}")
        return engine


# Instancier et entraîner
rag_engine = RAGEngine(
    top_k                = params['rag']['top_k'],
    similarity_threshold = params['rag']['similarity_threshold']
)
rag_engine.fit(rag_df)

In [ ]:
import sys
sys.path.insert(0, '../src')
from data.preprocess import clean_text

print("TESTS DU MOTEUR RAG")
print("=" * 60)

cas_tests = [
    {
        "titre"    : "Erreur activation Office 365",
        "desc"     : "Microsoft Office affiche message produit non activé",
        "categorie": "Microsoft"
    },
    {
        "titre"    : "Problème accès Sage X3",
        "desc"     : "Impossible de se connecter à l'ERP Sage X3 depuis ce matin",
        "categorie": "Applicatifs"
    },
    {
        "titre"    : "Sauvegarde échouée",
        "desc"     : "Le système de sauvegarde indique espace disque insuffisant",
        "categorie": "Backup"
    },
    {
        "titre"    : "Accès internet lent",
        "desc"     : "Connexion internet très lente depuis le bureau au 3ème étage",
        "categorie": "Réseau"
    }
]

for cas in cas_tests:
    query = clean_text(cas['desc'])
    recommendations = rag_engine.recommend(
        query_text      = query,
        category_filter = cas['categorie']
    )

    print(f"\n{'─'*60}")
    print(f"🎫 Ticket  : {cas['titre']}")
    print(f"📂 Catégo  : {cas['categorie']}")
    print(f"🔍 Query   : {query[:70]}...")
    print(f"💡 Recommandations ({len(recommendations)}) :")

    if not recommendations:
        print("   ⚠️  Aucune solution similaire trouvée")
    else:
        for i, rec in enumerate(recommendations, 1):
            print(f"\n   [{i}] Score     : {rec['score']:.4f}")
            print(f"       Résolution : {rec['resolution'][:100]}")
            print(f"       Knowledge  : {rec['knowledge'][:100]}")

In [ ]:
print("ÉVALUATION QUANTITATIVE DU RAG")
print("=" * 60)

# Pour chaque catégorie : taux de couverture (au moins 1 recommandation)
resultats_cat = {}

for cat in rag_df['Catégorie'].unique():
    tickets_cat = rag_df[rag_df['Catégorie'] == cat].sample(
        min(10, len(rag_df[rag_df['Catégorie'] == cat])),
        random_state=42
    )

    avec_reco = 0
    for _, row in tickets_cat.iterrows():
        query = str(row['text_rag'])
        recs  = rag_engine.recommend(query, category_filter=cat)
        if len(recs) > 0:
            avec_reco += 1

    taux = avec_reco / len(tickets_cat) * 100
    resultats_cat[cat] = {
        'tickets_testes' : len(tickets_cat),
        'avec_reco'      : avec_reco,
        'couverture_pct' : round(taux, 1)
    }

print(f"\n{'Catégorie':20s} | {'Testés':>8} | {'Avec reco':>10} | {'Couverture':>10}")
print("-" * 60)
for cat, res in resultats_cat.items():
    print(f"{cat:20s} | {res['tickets_testes']:>8} | "
          f"{res['avec_reco']:>10} | {res['couverture_pct']:>9.1f}%")

global_coverage = np.mean([r['couverture_pct'] for r in resultats_cat.values()])
print(f"\n✅ Couverture globale : {global_coverage:.1f}%")

In [ ]:
# Sauvegarder le moteur RAG
rag_engine.save('../models/rag_engine.pkl')

# Vérifier le rechargement
rag_loaded = RAGEngine.load('../models/rag_engine.pkl')
test_recs  = rag_loaded.recommend(
    clean_text("microsoft office erreur activation licence"),
    category_filter="Microsoft"
)
print(f"\n🧪 Test rechargement : {len(test_recs)} recommandation(s) trouvée(s)")